In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import trim_mean

data = {
    "mode": ["Bike"] * 10 + ["Taxi"] * 10 + ["Bus"] * 10,

    "wait_min": [
        2, 3, 3, 4, 4, 5, 5, 5, 6, 7,       # Bike
        4, 5, 6, 6, 7, 8, 8, 9, 12, 25,     # Taxi
        7, 8, 8, 9, 10, 11, 12, 14, 18, 41 # Bus
    ],

    "peak": ["No", "No", "Yes", "No", "Yes"] * 6
}

df = pd.DataFrame(data)

display(df.head())
print("Shape:", df.shape)

,mode,wait_min,peak
0,Bike,2,No
1,Bike,3,No
2,Bike,3,Yes
3,Bike,4,No
4,Bike,4,Yes


Shape: (30, 3)


In [ ]:
# Q1 — Absolute and Relative Frequency Tables

# Frequency distribution for mode
mode_freq = df["mode"].value_counts()

mode_relative = df["mode"].value_counts(
    normalize=True
)

mode_summary = pd.DataFrame({
    "Frequency": mode_freq,
    "Relative Frequency": mode_relative,
    "Percentage": mode_relative * 100
})

display(mode_summary.round(3))


# Frequency distribution for peak
peak_freq = df["peak"].value_counts()

peak_relative = df["peak"].value_counts(
    normalize=True
)

peak_summary = pd.DataFrame({
    "Frequency": peak_freq,
    "Relative Frequency": peak_relative,
    "Percentage": peak_relative * 100
})

display(peak_summary.round(3))

In [ ]:
# Q2 — Waiting-Time Bins and Cumulative Percentages

bins = np.arange(0, 46, 5)

df["wait_bin"] = pd.cut(
    df["wait_min"],
    bins=bins,
    right=False
)

freq = df["wait_bin"].value_counts(
    sort=False
)

frequency_table = pd.DataFrame({
    "Frequency": freq,
    "Relative Frequency": freq / len(df),
    "Cumulative Frequency": freq.cumsum(),
    "Cumulative Percentage": freq.cumsum() / len(df) * 100
})

display(frequency_table.round(2))

In [ ]:
# Q3 — Mean, Median, Mode and 10% Trimmed Mean

summary_center = df.groupby("mode").agg(
    mean_wait=("wait_min", "mean"),

    median_wait=("wait_min", "median"),

    mode_wait=(
        "wait_min",
        lambda x: x.mode().tolist()
    ),

    trimmed_mean=(
        "wait_min",
        lambda x: trim_mean(x, 0.1)
    )
)

display(summary_center.round(2))

# Bus has the largest difference between its mean and trimmed mean. 
# This suggests that extreme waiting times have a strong influence on the ordinary mean.

# Recall that a 10% trimmed mean removes the smallest 10% and largest 10% of observations before calculating the mean. 
# For 10 observations, this removes one observation from each end.

In [ ]:
# Q4 — Range, Sample SD, Q1, Q3 and IQR

summary_spread = df.groupby("mode").agg(
    minimum=("wait_min", "min"),
    maximum=("wait_min", "max"),
    range_wait=(
        "wait_min",
        lambda x: x.max() - x.min()
    ),
    sample_sd=("wait_min", "std"),
    q1=("wait_min", lambda x: x.quantile(0.25)),
    q3=("wait_min", lambda x: x.quantile(0.75)),
    iqr=("wait_min", lambda x: x.quantile(0.75) - x.quantile(0.25))
)

display(summary_spread.round(2))

In [ ]:
# Q5 — Difference Between Mean and Median

difference = summary_center.copy()

difference["mean_median_diff"] = (
    difference["mean_wait"] -
    difference["median_wait"]
)

difference["absolute_diff"] = (
    difference["mean_median_diff"].abs()
)

display(
    difference[
        ["mean_wait", "median_wait",
         "mean_median_diff", "absolute_diff"]
    ].round(2)
)

largest_group = difference["absolute_diff"].idxmax()

print("Largest difference:", largest_group)

# Bus shows the largest difference between mean and median. Its mean is pulled upward by unusually long waiting times, 
# especially the 41-minute observation, suggesting a right-skewed distribution.

In [ ]:
# Q6 — Histogram and Side-by-Side Boxplots

plt.figure(figsize=(9, 5))

plt.hist(
    df["wait_min"],
    bins=np.arange(0, 46, 5),
    edgecolor="white"
)

plt.title("Distribution of Passenger Waiting Times")
plt.xlabel("Waiting Time (minutes)")
plt.ylabel("Frequency")
plt.xticks(np.arange(0, 46, 5))

plt.show()

In [ ]:
# Side-by-side boxplots

bike = df[df["mode"] == "Bike"]["wait_min"]
taxi = df[df["mode"] == "Taxi"]["wait_min"]
bus = df[df["mode"] == "Bus"]["wait_min"]

plt.figure(figsize=(8, 5))

plt.boxplot(
    [bike, taxi, bus]
)

plt.xticks([1, 2, 3], ["Bike", "Taxi", "Bus"])

plt.title("Waiting Times by Transportation Mode")
plt.xlabel("Transportation Mode")
plt.ylabel("Waiting Time (minutes)")

plt.show()

Interpretation:

# The plots support the numerical results. Bike waiting times are concentrated in a narrow interval. 
# Taxi has a longer right tail, while Bus shows the greatest variability and an extreme 41-minute wait.

In [ ]:
# Q7 — Mean + SD or Median + IQR?

final_summary = summary_center.join(
    summary_spread
)

display(
    final_summary[
        [
            "mean_wait",
            "median_wait",
            "sample_sd",
            "iqr",
            "trimmed_mean"
        ]
    ].round(2)
)

Q8 — Operational Summary (120–180 words)


The dataset contains 30 passenger waiting-time observations across three transportation modes: Bike, Taxi, and Bus, with ten observations per mode. Bike has the shortest typical waiting time, with a mean of 4.4 minutes and a median of 4.5 minutes. Its relatively small standard deviation indicates consistent service.

Taxi has a mean waiting time of 9.0 minutes, compared with a median of 7.5 minutes. This difference reflects a longer right tail, including one 25-minute wait. Bus has the highest mean waiting time at 13.8 minutes and the greatest variability, with a standard deviation of approximately 10.11 minutes. A 41-minute wait substantially influences its mean.

For operational reporting, mean and standard deviation are reasonable for Bike, while median and IQR are more representative for Taxi and Bus. The unusually long waiting times should be investigated rather than automatically removed. However, the sample is small, so these findings should not be generalized without additional data.

In [2]:
from scipy.stats import trim_mean

final_stats = df.groupby("mode").agg(
    n=("wait_min", "count"),
    mean=("wait_min", "mean"),
    median=("wait_min", "median"),
    mode=("wait_min", lambda x: x.mode().tolist()),
    trimmed_mean=("wait_min", lambda x: trim_mean(x, 0.1)),
    sd=("wait_min", "std"),
    minimum=("wait_min", "min"),
    maximum=("wait_min", "max"),
    q1=("wait_min", lambda x: x.quantile(0.25)),
    q3=("wait_min", lambda x: x.quantile(0.75))
)

final_stats["range"] = (
    final_stats["maximum"] - final_stats["minimum"]
)

final_stats["iqr"] = (
    final_stats["q3"] - final_stats["q1"]
)

final_stats["mean_median_gap"] = (
    final_stats["mean"] - final_stats["median"]
)

display(final_stats.round(2))

,n,mean,median,mode,trimmed_mean,sd,minimum,maximum,q1,q3,range,iqr,mean_median_gap
mode,,,,,,,,,,,,,
Bike,10,4.4,4.5,[5],4.38,1.51,2,7,3.25,5.00,5,1.75,-0.1
Bus,10,13.8,10.5,[8],11.25,10.11,7,41,8.25,13.50,34,5.25,3.3
Taxi,10,9.0,7.5,"[6, 8]",7.62,6.06,4,25,6.00,8.75,21,2.75,1.5
